In [30]:
import pandas as pd
train_df = pd.read_csv("../data/raw/Train.csv")
test_df = pd.read_csv("../data/raw/Test.csv")


### Defining feature groups

In [31]:
TARGET = 'bank_account'

binary_features = [
    'cellphone_access',
    'gender_of_respondent',
    'location_type'
]

ordinal_features = ['education_level']

nominal_features = [
    'job_type',
    'marital_status',
    'relationship_with_head',
    'country'
]

numeric_features = [
    'age_of_respondent',
    'household_size'
]


### Designing encoders

In [32]:
from sklearn.preprocessing import FunctionTransformer

def binary_encode(df):
    return df.replace({
        'Yes': 1, 'No': 0,
        'Male': 1, 'Female': 0,
        'Urban': 1, 'Rural': 0
    })

binary_mapper = FunctionTransformer(binary_encode)



**Ordinal encoding** 

In [33]:
from sklearn.preprocessing import OrdinalEncoder

education_order = [[
    "No formal education",
    "Primary education",
    "Secondary education",
    "Vocational/Specialised training",
    "Tertiary education",
    "Other/Dont know/RTA"
]]

ordinal_encoder = OrdinalEncoder(categories=education_order)


**One-hot encoding**

In [34]:
from sklearn.preprocessing import OneHotEncoder

nominal_encoder = OneHotEncoder(
    handle_unknown='ignore',
    drop='first'
)


### Numeric transformations

In [35]:
from sklearn.preprocessing import StandardScaler

numeric_transformer = StandardScaler()


### deriving features

In [36]:
def add_derived_features(df):
    df = df.copy()

    df['is_head_or_spouse'] = df['relationship_with_head'].isin(
        ['Head of Household', 'Spouse']
    ).astype(int)

    df['high_education'] = df['education_level'].isin(
        ['Secondary education',
         'Vocational/Specialised training',
         'Tertiary education']
    ).astype(int)

    df['urban'] = (df['location_type'] == 'Urban').astype(int)

    return df


**Test on both datasets, train and test**

In [37]:
train_df = add_derived_features(train_df)
test_df = add_derived_features(test_df)


### Build preprocessing pipeline

In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ('bin', binary_mapper, binary_features),
        ('ord', ordinal_encoder, ordinal_features),
        ('nom', nominal_encoder, nominal_features),
        ('num', numeric_transformer, numeric_features)
    ]
)


### Sanity checking

In [39]:
X = train_df.drop(columns=TARGET)

preprocessor.fit(X)
X_transformed = preprocessor.transform(X)

X_transformed.shape


(23524, 27)

### Saving logic for reusing

In [40]:
import joblib

joblib.dump(preprocessor, "../artifacts/preprocessor.pkl")


['../artifacts/preprocessor.pkl']